In [ ]:
import importlib
from models.API import GeneratorAPI
import torch
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
print(f"CUDA_LAUNCH_BLOCKING={os.environ.get('CUDA_LAUNCH_BLOCKING')}")

# ──── SWITCH: "dj30" or "etf" ────
MODE = "dj30"

if MODE == "dj30":
    model_path = "output/dj30"
    ticker_name = "AAPL"
    obs_features = [
        'open', 'high', 'low', 'close', 'adj_close',
        'kmid', 'kmid2', 'klen', 'kup', 'kup2',
        'klow', 'klow2', 'ksft', 'ksft2',
        'roc_5', 'roc_10', 'roc_20', 'roc_30', 'roc_60',
        'ma_5', 'ma_10', 'ma_20', 'ma_30', 'ma_60',
        'std_5', 'std_10', 'std_20', 'std_30', 'std_60',
        'beta_5', 'beta_10', 'beta_20', 'beta_30', 'beta_60',
        'max_5', 'max_10', 'max_20', 'max_30', 'max_60',
        'min_5', 'min_10', 'min_20', 'min_30', 'min_60',
        'qtlu_5', 'qtlu_10', 'qtlu_20', 'qtlu_30', 'qtlu_60',
        'qtld_5', 'qtld_10', 'qtld_20', 'qtld_30', 'qtld_60',
        'rank_5', 'rank_10', 'rank_20', 'rank_30', 'rank_60',
        'rsv_5', 'rsv_10', 'rsv_20', 'rsv_30', 'rsv_60',
        'imax_5', 'imax_10', 'imax_20', 'imax_30', 'imax_60',
        'imin_5', 'imin_10', 'imin_20', 'imin_30', 'imin_60',
        'imxd_5', 'imxd_10', 'imxd_20', 'imxd_30', 'imxd_60',
        'corr_5', 'corr_10', 'corr_20', 'corr_30', 'corr_60',
        'cord_5', 'cord_10', 'cord_20', 'cord_30', 'cord_60',
        'cntp_5', 'cntp_10', 'cntp_20', 'cntp_30', 'cntp_60',
        'cntn_5', 'cntn_10', 'cntn_20', 'cntn_30', 'cntn_60',
        'cntd_5', 'cntd_10', 'cntd_20', 'cntd_30', 'cntd_60',
        'sump_5', 'sump_10', 'sump_20', 'sump_30', 'sump_60',
        'sumn_5', 'sumn_10', 'sumn_20', 'sumn_30', 'sumn_60',
        'sumd_5', 'sumd_10', 'sumd_20', 'sumd_30', 'sumd_60',
        'vma_5', 'vma_10', 'vma_20', 'vma_30', 'vma_60',
        'vstd_5', 'vstd_10', 'vstd_20', 'vstd_30', 'vstd_60',
        'wvma_5', 'wvma_10', 'wvma_20', 'wvma_30', 'wvma_60',
        'vsump_5', 'vsump_10', 'vsump_20', 'vsump_30', 'vsump_60',
        'vsumn_5', 'vsumn_10', 'vsumn_20', 'vsumn_30', 'vsumn_60',
        'vsumd_5', 'vsumd_10', 'vsumd_20', 'vsumd_30', 'vsumd_60',
    ]
    temporal_features = ['day_of_week', 'day_of_month', 'month_of_year']
elif MODE == "etf":
    model_path = "output/etf_6_120"
    ticker_name = "CORN"
    obs_features = None      # fill in ETF feature list if needed
    temporal_features = None  # fill in ETF temporals if needed
else:
    raise ValueError(f"Unknown MODE: {MODE}")

print(f"Mode: {MODE} | Model: {model_path} | Ticker: {ticker_name}")

# Reload module to pick up any API.py changes
importlib.reload(importlib.import_module("models.API"))
GeneratorAPI = importlib.import_module("models.API").GeneratorAPI

api = GeneratorAPI(model_path=model_path, ticker_name=ticker_name,
                   obs_features=obs_features, temporal_features=temporal_features)

CUDA_LAUNCH_BLOCKING=1
Using GPUs: 3
All data and model initialized successfully.


/data3/hcxia/Adahist2/generator/GRT_GAN/models/API.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f"{model_path}/model.pt")



Generator loaded


In [ ]:
# ──── Diagnose date range ────
import numpy as np, pandas as pd

sd = np.load('../../datasets/output_data/output_starting_date.npy', allow_pickle=True)
gan_dates = pd.DatetimeIndex(sd[:, 0])
print(f"GAN dates: {gan_dates[0].date()} → {gan_dates[-1].date()}  ({len(gan_dates)} samples)")

# Check RL parquet dates for the selected ticker
parquet_path = f"../../datasets/processd_day_dj30/features/{ticker_name}.parquet" if MODE == "dj30" \
    else f"../../datasets/processd_day_future_etfs/features/{ticker_name}.parquet"
try:
    df = pd.read_parquet(parquet_path)
    if 'timestamp' in df.columns:
        df = df.set_index('timestamp')
    rl_dates = df.index
    overlap = gan_dates.intersection(rl_dates)
    print(f"RL dates:  {rl_dates[0].date()} → {rl_dates[-1].date()}  ({len(rl_dates)} dates)")
    print(f"Overlap:   {len(overlap)} dates")
    print(f"RL dates before GAN start: {(rl_dates < gan_dates[0]).sum()}")
    print(f"RL dates after GAN end:    {(rl_dates > gan_dates[-1]).sum()}")
except FileNotFoundError:
    print(f"Parquet not found: {parquet_path}")

In [ ]:
# ──── Test api.call() with various timestamps ────
import numpy as np, pandas as pd

macro_epsilon = np.random.randn(46)

# Test 1: Date inside GAN range (should work)
ts_in = "2011-01-03"
feature = api.call(pd.Timestamp(ts_in), macro_epsilon)
print(f"✓ In-range  ({ts_in}): shape={feature.shape}")

# Test 2: Date BEFORE GAN range (should use nearest = first GAN date)
ts_before = "2001-06-15"
feature2 = api.call(pd.Timestamp(ts_before), macro_epsilon)
print(f"✓ Pre-range ({ts_before}): shape={feature2.shape}  (nearest-date fallback)")

# Test 3: Date AFTER GAN range (should use nearest = last GAN date)
ts_after = "2023-06-15"
feature3 = api.call(pd.Timestamp(ts_after), macro_epsilon)
print(f"✓ Post-range({ts_after}): shape={feature3.shape}  (nearest-date fallback)")

['CORN' 'CYB' 'DBB' 'DBC' 'FXA' 'FXB' 'FXC' 'FXE' 'FXY' 'GLD' 'IWM' 'NIB'
 'QQQ' 'SLV' 'SPY' 'UGA' 'UNG' 'USO']
ticker_index:  0
real_data:  (1, 120, 90)
target_macro:  (1, 120, 46)
processed_pv_feature all shape:  (1, 120, 90)
processed_pv_feature all shape:  (120, 90)
processed_pv_feature shape:  (120, 5)
open all shape:  (2397, 18)
close all shape:  (2397, 18, 120)
close shape:  ()
close 12.40999984741211
open 12.210000038146973
df_features shape:  (120, 5)
original_close:  12.40999984741211
original_open:  12.210000038146973
smoothed_close shape:  (120,)
smoothed_open shape:  (120,)
smoothed_high shape:  (120,)
smoothed_low shape:  (120,)
open:  0      1.221000e+01
1      1.383760e+00
2      9.394402e+00
3      2.121274e+01
4     -4.958565e+00
           ...     
115   -1.174590e-16
116   -1.755200e-16
117   -4.982822e-16
118   -5.558472e-16
119   -1.336656e-15
Length: 120, dtype: float64
close:  0      1.241000e+01
1      1.441336e+01
2      9.293713e+00
3      1.930116e+01
4     

/data3/hcxia/Adahist2/generator/GRT_GAN/models/API.py:275: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["cord_{}".format(w)] = df1.rolling(w).corr(pairwise = df2.rolling(w))
/data3/hcxia/Adahist2/generator/GRT_GAN/models/API.py:277: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['abs_ret1'] = np.abs(df['ret1'])
/data3/hcxia/Adahist2/generator/GRT_GAN/models/API.py:278: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider 

In [ ]:
# ──── Inspect generated features ────
print(f"Feature shape: {feature.shape}")
print(f"Columns: {list(feature.columns[:10])}...")
print(f"\nSample stats (in-range {ts_in}):")
print(feature.describe().iloc[:4, :6])  # mean/std/min/max for first 6 cols

[[Timestamp('2011-01-03 00:00:00') Timestamp('2011-01-03 00:00:00')
  Timestamp('2011-01-03 00:00:00') ... Timestamp('2011-01-03 00:00:00')
  Timestamp('2011-01-03 00:00:00') Timestamp('2011-01-03 00:00:00')]
 [Timestamp('2011-01-04 00:00:00') Timestamp('2011-01-04 00:00:00')
  Timestamp('2011-01-04 00:00:00') ... Timestamp('2011-01-04 00:00:00')
  Timestamp('2011-01-04 00:00:00') Timestamp('2011-01-04 00:00:00')]
 [Timestamp('2011-01-05 00:00:00') Timestamp('2011-01-05 00:00:00')
  Timestamp('2011-01-05 00:00:00') ... Timestamp('2011-01-05 00:00:00')
  Timestamp('2011-01-05 00:00:00') Timestamp('2011-01-05 00:00:00')]
 ...
 [Timestamp('2020-07-09 00:00:00') Timestamp('2020-07-09 00:00:00')
  Timestamp('2020-07-09 00:00:00') ... Timestamp('2020-07-09 00:00:00')
  Timestamp('2020-07-09 00:00:00') Timestamp('2020-07-09 00:00:00')]
 [Timestamp('2020-07-10 00:00:00') Timestamp('2020-07-10 00:00:00')
  Timestamp('2020-07-10 00:00:00') ... Timestamp('2020-07-10 00:00:00')
  Timestamp('2020-0